# PID — Fine-tuning en M3FD (Kaggle T4)

**Modelo:** Physics-Informed Diffusion para generación de imágenes infrarrojas  
**Dataset:** M3FD (Multi-Modal Multi-Spectral Fusion Dataset)  
**GPU Kaggle:** Tesla T4 · 16 GB VRAM  

### Datasets que este notebook necesita conectados
Antes de correr, en el panel derecho de Kaggle → **Add Data**:
1. `m3fd-dataset` — el dataset M3FD con carpetas `Vis/`, `Ir/`, `Annotation/`
2. `pid-checkpoints` — los pesos pre-entrenados que subiste (ver celda 2 para estructura)
3. `pid-repo` — el código del repositorio PID (zip del fork)

### Parámetros ajustados vs entrenamiento local
| Parámetro | Local RTX 4050 | Kaggle T4 | Por qué cambia |
|-----------|---------------|-----------|----------------|
| `batch_size` | 4 | 12 | T4 tiene 16 GB vs 6 GB |
| `num_workers` | 2 | 4 | Linux Kaggle soporta fork |
| `accumulate_grad_batches` | 2 | 1 | No hace falta compensar en T4 |
| `image_size` | 512 | 512 | Igual — no cambia la arquitectura |

In [ ]:
# ── Celda 1: Verificar GPU ────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
# ── Celda 2: Verificar estructura de los datasets conectados ──────────────
import os

M3FD_BASE   = '/kaggle/input/m3fd-dataset'
CKPT_BASE   = '/kaggle/input/pid-checkpoints'
REPO_BASE   = '/kaggle/input/pid-repo'
WORK_DIR    = '/kaggle/working/PID'

print('=== M3FD dataset ===')
for folder in ['Vis', 'Ir', 'Annotation']:
    path = os.path.join(M3FD_BASE, folder)
    if os.path.exists(path):
        n = len(os.listdir(path))
        print(f'  {folder}/  →  {n} archivos  ✓')
    else:
        print(f'  {folder}/  →  NO ENCONTRADO ✗')

print('\n=== Checkpoints ===')
for f in [
    'vqf8_pretrained/model.ckpt',
    'TeVNet_KAIST/epoch_950.pth',
    'PID_KAIST/last.ckpt',
]:
    path = os.path.join(CKPT_BASE, f)
    exists = os.path.exists(path)
    size_mb = round(os.path.getsize(path)/1e6) if exists else 0
    print(f'  {f}  →  {size_mb} MB  {"✓" if exists else "✗"}')

In [ ]:
# ── Celda 3: Instalar dependencias ────────────────────────────────────────
# Kaggle ya tiene torch/torchvision instalados con CUDA
# Solo instalamos las dependencias específicas del proyecto

!pip install -q \
    pytorch-lightning==1.4.2 \
    omegaconf \
    einops \
    kornia \
    "transformers<5" \
    segmentation-models-pytorch \
    "torchmetrics==0.6.0" \
    "setuptools<71" \
    imageio

print('Dependencias instaladas.')

In [ ]:
# ── Celda 4: Copiar repo y configurar rutas ───────────────────────────────
import shutil, os

# Descomprimir o copiar el repo a /kaggle/working/
if not os.path.exists(WORK_DIR):
    # Si subiste el repo como .zip
    zip_path = os.path.join(REPO_BASE, 'PID.zip')
    if os.path.exists(zip_path):
        shutil.unpack_archive(zip_path, '/kaggle/working/')
    else:
        # Si subiste la carpeta directamente
        shutil.copytree(REPO_BASE, WORK_DIR)

os.chdir(WORK_DIR)
print('Directorio de trabajo:', os.getcwd())
print('Contenido:', os.listdir('.'))

In [ ]:
# ── Celda 5: Resolver mismatch de rutas M3FD ─────────────────────────────
# El dataset M3FD en Kaggle usa Vis/ e Ir/ (mayúsculas)
# Nuestro M3FD.py espera vi/ e ir/ (minúsculas)
# Solución: crear symlinks en /kaggle/working/ sin copiar los datos

M3FD_WORK = os.path.join(WORK_DIR, 'M3FD')
os.makedirs(M3FD_WORK, exist_ok=True)

vi_link = os.path.join(M3FD_WORK, 'vi')
ir_link  = os.path.join(M3FD_WORK, 'ir')

if not os.path.exists(vi_link):
    os.symlink(os.path.join(M3FD_BASE, 'Vis'), vi_link)
if not os.path.exists(ir_link):
    os.symlink(os.path.join(M3FD_BASE, 'Ir'), ir_link)

print(f'vi/ → {os.readlink(vi_link)}')
print(f'ir/ → {os.readlink(ir_link)}')
print(f'Imágenes IR  : {len(os.listdir(ir_link))}')
print(f'Imágenes Vis : {len(os.listdir(vi_link))}')

In [ ]:
# ── Celda 6: Generar split train/val/test (seed=42) ───────────────────────
import random
from pathlib import Path

def generate_split(ir_dir, out_dir, train=0.70, val=0.15, seed=42):
    names = sorted(Path(ir_dir).stem for p in Path(ir_dir).glob('*.png') for _ in [None]
                   if (names := p.stem) or True)  # comprehension trick
    # Versión limpia:
    names = sorted(p.stem for p in Path(ir_dir).glob('*.png'))
    if not names:
        names = sorted(p.stem for p in Path(ir_dir).glob('*.jpg'))

    rng = random.Random(seed)
    rng.shuffle(names)

    n = len(names)
    n_train = int(n * train)
    n_val   = int(n * val)

    splits = {
        'train': names[:n_train],
        'val':   names[n_train:n_train + n_val],
        'test':  names[n_train + n_val:],
    }

    Path(out_dir).mkdir(parents=True, exist_ok=True)
    for split_name, split_list in splits.items():
        (Path(out_dir) / f'{split_name}.txt').write_text('\n'.join(split_list) + '\n')
        print(f'  {split_name}: {len(split_list)} imágenes')

    return splits

splits = generate_split(
    ir_dir  = os.path.join(M3FD_WORK, 'ir'),
    out_dir = os.path.join(WORK_DIR, 'data/M3FD'),
)
print(f'\nTotal: {sum(len(v) for v in splits.values())} imágenes')

In [ ]:
# ── Celda 7: Crear symlinks para checkpoints ──────────────────────────────
import shutil

PRETRAINED = os.path.join(WORK_DIR, 'pretrained')

links = {
    'vqf8_pretrained/model.ckpt':   os.path.join(CKPT_BASE, 'vqf8_pretrained/model.ckpt'),
    'TeVNet_KAIST/epoch_950.pth':   os.path.join(CKPT_BASE, 'TeVNet_KAIST/epoch_950.pth'),
    'PID_KAIST/last.ckpt':          os.path.join(CKPT_BASE, 'PID_KAIST/last.ckpt'),
}

for rel_dst, src in links.items():
    dst = os.path.join(PRETRAINED, rel_dst)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not os.path.exists(dst):
        os.symlink(src, dst)
    size_mb = round(os.path.getsize(src) / 1e6)
    print(f'  {rel_dst}  ({size_mb} MB)  ✓')

In [ ]:
# ── Celda 8: Verificar que el dataloader M3FD carga correctamente ─────────
import sys
sys.path.insert(0, WORK_DIR)

from torch.utils.data import DataLoader
from ldm.data.M3FD import M3FDTrain, M3FDVal

train_ds = M3FDTrain(size=512)
val_ds   = M3FDVal(size=512)

loader = DataLoader(train_ds, batch_size=2, num_workers=0)
batch  = next(iter(loader))

print(f'Train: {len(train_ds)} imágenes')
print(f'Val  : {len(val_ds)} imágenes')
print(f'image shape      : {list(batch["image"].shape)}')
print(f'conditional shape: {list(batch["conditional"].shape)}')
print(f'rango imagen     : [{batch["image"].min():.2f}, {batch["image"].max():.2f}]')
print('Dataloader OK ✓')

In [ ]:
# ── Celda 9: Escribir config YAML para Kaggle T4 ─────────────────────────
# Ajustes respecto a la config local:
#   batch_size: 4 → 12   (T4 tiene 16 GB, el modelo necesita ~1.2 GB/muestra)
#   num_workers: 2 → 4   (Linux Kaggle soporta multiprocessing con fork)

yaml_content = """
model:
  base_learning_rate: 1.0e-06
  target: ldm.models.diffusion.ddpm_tev.LatentDiffusion
  params:
    load_only_unet: True
    tevloss_weight_rec: 0
    tevloss_weight_tev: 0
    pixel_tev: true
    vnums: 4
    linear_start: 0.0015
    linear_end: 0.0205
    log_every_t: 100
    timesteps: 1000
    loss_type: l1
    first_stage_key: image
    cond_stage_key: conditional
    image_size: 64
    channels: 4
    concat_mode: true
    monitor: val/loss_simple_ema
    cond_stage_trainable: true
    unet_config:
      target: ldm.modules.diffusionmodules.openaimodel.UNetModel
      params:
        image_size: 64
        in_channels: 7
        out_channels: 4
        model_channels: 128
        attention_resolutions: [8, 4, 2]
        num_res_blocks: 2
        channel_mult: [1, 4, 8]
        num_head_channels: 8
    first_stage_config:
      target: ldm.models.autoencoder.VQModelInterface
      params:
        ckpt_path: "./pretrained/vqf8_pretrained/model.ckpt"
        embed_dim: 4
        n_embed: 16384
        monitor: val/rec_loss
        ddconfig:
          double_z: false
          z_channels: 4
          resolution: 256
          in_channels: 3
          out_ch: 3
          ch: 128
          ch_mult: [1, 2, 2, 4]
          num_res_blocks: 2
          attn_resolutions: [32]
          dropout: 0
        lossconfig:
          target: torch.nn.Identity
    cond_stage_config:
      target: ldm.modules.encoders.modules.SpatialRescaler
      params:
        n_stages: 3
        method: bicubic
        in_channels: 3
        out_channels: 3
    tev_net_config:
      target: ldm.modules.HADARNet.modules.HADARNet
      params:
        in_channels: 3
        out_channels: 6
        smp_model: Unet
        smp_encoder: resnet18
        ckpt_path: "./pretrained/TeVNet_KAIST/epoch_950.pth"

data:
  target: main.DataModuleFromConfig
  params:
    batch_size: 12
    num_workers: 4
    wrap: false
    train:
      target: ldm.data.M3FD.M3FDTrain
      params:
        size: 512
        flip_p: 0.5
        crop_p: 0.5
    validation:
      target: ldm.data.M3FD.M3FDVal
      params:
        size: 512
"""

cfg_path = os.path.join(WORK_DIR, 'configs/latent-diffusion/PID-M3FD-kaggle.yaml')
with open(cfg_path, 'w') as f:
    f.write(yaml_content.strip())
print('Config YAML escrita en:', cfg_path)

In [ ]:
# ── Celda 10: ENTRENAMIENTO ───────────────────────────────────────────────
# Kaggle permite hasta 12h de GPU por sesión.
# Con batch_size=12 y ~4200 imágenes de train, cada epoch toma ~8 min en T4.
# En 12h entran aproximadamente 80-90 epochs.

import subprocess, sys

cmd = [
    sys.executable, 'main.py',
    '-t', 'True',
    '-b', 'configs/latent-diffusion/PID-M3FD-kaggle.yaml',
    '--gpus', '0,',
    f'model.params.ckpt_path=pretrained/PID_KAIST/last.ckpt',
    '--max_epochs', '100',
]

print('Comando:', ' '.join(cmd))
print('Iniciando entrenamiento...')

result = subprocess.run(cmd, cwd=WORK_DIR)
print('\nCódigo de salida:', result.returncode)

In [ ]:
# ── Celda 11: Guardar el checkpoint final en /kaggle/working/ ─────────────
# Kaggle guarda automáticamente todo lo que esté en /kaggle/working/
# al terminar la sesión. Los logs y checkpoints quedan en logs/

import glob

checkpoints = glob.glob(os.path.join(WORK_DIR, 'logs/**/checkpoints/*.ckpt'), recursive=True)
print(f'Checkpoints guardados ({len(checkpoints)}):')
for c in sorted(checkpoints):
    size_mb = round(os.path.getsize(c) / 1e6)
    print(f'  {os.path.relpath(c, WORK_DIR)}  ({size_mb} MB)')